<a target="_blank" href="https://colab.research.google.com/github/sandrozc-stripe/demo-dev-value-workshop/blob/main/colab-notebooks/01_api_testing_intro.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 01: API Testing — Introduction

**Duration**: ~10 minutes

## Learning Objectives

By the end of this notebook, you will:
- Understand Stripe sandboxes and when to use them
- Use the `expand` parameter to retrieve related objects in one API call
- Store custom data on Stripe objects with `metadata`
- Know the three testing tools available: Sandboxes, Test Cards, and Test Clocks

---

## Step 1: Install Dependencies

Run the cell below to install the Stripe Python library.

In [17]:
!pip install stripe --quiet
print("Dependencies installed.")

Dependencies installed.


## Step 2: Configure Your API Key

**Recommended (Colab Secrets):**
1. Click the **key icon** in the left sidebar
2. Add a secret named `STRIPE_SECRET_KEY` with your test key (`sk_test_...`)
3. Toggle **Notebook access** to on

If you skip that, the cell below will prompt you to paste your key.

In [18]:
import os
import stripe

# Load API key from Colab Secrets or prompt
try:
    from google.colab import userdata
    STRIPE_SECRET_KEY = userdata.get('STRIPE_SECRET_KEY')
    print("Loaded API key from Colab Secrets.")
except Exception:
    import getpass
    STRIPE_SECRET_KEY = getpass.getpass("Paste your Stripe test secret key (sk_test_...): ")

stripe.api_key = STRIPE_SECRET_KEY

# Safety check
if stripe.api_key and 'test' in stripe.api_key:
    print("Safe: Using SANDBOX API key.")
elif stripe.api_key:
    print("WARNING: This looks like a LIVE key. Use a test key (sk_test_...) for this workshop!")
else:
    print("ERROR: No API key provided.")

# Verify the connection
try:
    account = stripe.Account.retrieve()
    print(f"Connected to Stripe account: {account.id}")
except stripe.error.AuthenticationError:
    print("ERROR: Invalid API key. Double-check your key and try again.")

Loaded API key from Colab Secrets.
Safe: Using SANDBOX API key.
Connected to Stripe account: acct_1RnL4mBMxfUzotEq


# New section

---

## Why Testing Matters

Testing your Stripe integration before going live prevents real money from moving, real cards from being charged, and real data from being affected.

| Scenario | Risk of skipping tests |
|----------|------------------------|
| Untested payment flow | Customers can't complete purchases |
| No error handling | Poor user experience on declines |
| Subscription bugs | Incorrect billing, lost revenue |
| Missing webhooks | Out-of-sync data, manual fixes |

## Stripe Sandboxes

[Sandboxes](https://docs.stripe.com/sandboxes) are isolated test environments.

| Feature | Detail |
|---------|--------|
| Up to 5 sandboxes | One per team, project, or partner |
| Granular access control | Choose who can access each sandbox |
| Fully isolated settings | No impact on live mode |
| V1 and V2 API support | Test the latest API features |

## API Key Prefixes

| Prefix | Type | Environment |
|--------|------|-------------|
| `sk_test_` | Secret key | Sandbox |
| `pk_test_` | Publishable key | Sandbox |
| `sk_live_` | Secret key | Live mode |
| `pk_live_` | Publishable key | Live mode |

**Never use live keys (`sk_live_`) for testing.** The [Stripe Services Agreement](https://stripe.com/legal/ssa) prohibits testing with real payment methods in live mode.

---

## The `expand` Parameter

By default, Stripe API responses return IDs for related objects rather than the full objects. The `expand` parameter retrieves related objects in a single API call.

```python
# WITHOUT expand — payment_method is just an ID string
pi = stripe.PaymentIntent.retrieve("pi_xxx")
print(pi.payment_method)  # "pm_xxx"

# WITH expand — payment_method is the full object
pi = stripe.PaymentIntent.retrieve("pi_xxx", expand=["payment_method"])
print(pi.payment_method.card.brand)  # "visa"
```

In [19]:
# Create a PaymentIntent to demonstrate expand
payment_intent = stripe.PaymentIntent.create(
    amount=1000,
    currency="usd",
    payment_method="pm_card_visa",
    confirm=True,
    automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
)

print("WITHOUT expand:")
print(f"  payment_method value: {payment_intent.payment_method}")
print(f"  Type: {type(payment_intent.payment_method).__name__}")

WITHOUT expand:
  payment_method value: pm_1TMUWeBMxfUzotEqYe1lPXk7
  Type: str


In [20]:
# Now retrieve the same PaymentIntent WITH expand
pi_expanded = stripe.PaymentIntent.retrieve(
    payment_intent.id,
    expand=["payment_method", "latest_charge"]
)

print("WITH expand:")
print(f"  payment_method type: {type(pi_expanded.payment_method).__name__}")
print(f"  Card brand: {pi_expanded.payment_method.card.brand}")
print(f"  Card last4: {pi_expanded.payment_method.card.last4}")
print(f"  Charge ID: {pi_expanded.latest_charge.id}")
print(f"  Charge amount: ${pi_expanded.latest_charge.amount / 100:.2f}")

WITH expand:
  payment_method type: PaymentMethod
  Card brand: visa
  Card last4: 4242
  Charge ID: ch_3TMUWeBMxfUzotEq4LQNWl1b
  Charge amount: $10.00


### Checkpoint

You should see:
- Without `expand`: `payment_method` is a plain string ID
- With `expand`: `payment_method` is a full object with card details

### Common Expandable Fields

| Object | Expandable fields |
|--------|-------------------|
| PaymentIntent | `payment_method`, `customer`, `latest_charge`, `invoice` |
| Subscription | `customer`, `default_payment_method`, `latest_invoice` |
| Invoice | `customer`, `subscription`, `charge`, `payment_intent` |
| Charge | `customer`, `payment_intent`, `balance_transaction` |

---

## The `metadata` Field

`metadata` lets you attach up to 50 custom key-value pairs to most Stripe objects. Use it to:
- Link Stripe objects to your internal IDs (order IDs, user IDs)
- Store business context (campaign source, shipping method)

**Limits:** up to 50 keys, keys max 40 chars, values max 500 chars, all values are strings.

In [21]:
import stripe

# Create a customer with metadata
customer = stripe.Customer.create(
    email="workshop-demo@example.com",
    name="Workshop Demo User",
    metadata={
        "internal_user_id": "usr_12345",
        "plan_tier": "enterprise",
        "signup_source": "workshop_demo",
        "account_manager": "jane.doe"
    }
)

print(f"Customer ID: {customer.id}")
print(f"Email: {customer.email}")
print("\nMetadata:")
# Use .to_dict() to explicitly convert StripeObject metadata to a Python dictionary
for key, value in customer.metadata.to_dict().items():
    print(f"  {key}: {value}")

Customer ID: cus_ULAsinNVwk3tLE
Email: workshop-demo@example.com

Metadata:
  account_manager: jane.doe
  internal_user_id: usr_12345
  plan_tier: enterprise
  signup_source: workshop_demo


In [22]:
# Create a PaymentIntent with order-tracking metadata
order_payment = stripe.PaymentIntent.create(
    amount=4999,
    currency="usd",
    customer=customer.id,
    payment_method="pm_card_visa",
    confirm=True,
    automatic_payment_methods={"enabled": True, "allow_redirects": "never"},
    metadata={
        "order_id": "ORD-2024-001234",
        "product_sku": "WIDGET-PRO-001",
        "shipping_method": "express",
        "promo_code": "WORKSHOP20"
    }
)

print(f"Payment ID: {order_payment.id}")
print(f"Amount: ${order_payment.amount / 100:.2f}")
print(f"Status: {order_payment.status}")
print("\nOrder Metadata:")
for key, value in order_payment.metadata.to_dict().items():
    print(f"  {key}: {value}")

Payment ID: pi_3TMUWfBMxfUzotEq0Vm5u1iH
Amount: $49.99
Status: succeeded

Order Metadata:
  order_id: ORD-2024-001234
  product_sku: WIDGET-PRO-001
  promo_code: WORKSHOP20
  shipping_method: express


In [23]:
# Update metadata on an existing customer
# - Provide a new value to update an existing key
# - Provide an empty string "" to remove a key
updated_customer = stripe.Customer.modify(
    customer.id,
    metadata={
        "plan_tier": "premium",       # Update existing key
        "upgraded_at": "2026-01-15",  # Add new key
        "signup_source": ""           # Empty string removes the key
    }
)

print("Updated Metadata:")
for key, value in updated_customer.metadata.to_dict().items():
    print(f"  {key}: {value}")

print("\nNote: signup_source was removed (set to empty string).")

Updated Metadata:
  account_manager: jane.doe
  internal_user_id: usr_12345
  plan_tier: premium
  upgraded_at: 2026-01-15

Note: signup_source was removed (set to empty string).


### Checkpoint

You should see:
- Customer created with 4 metadata fields
- PaymentIntent with order-tracking metadata
- Updated customer: `signup_source` is gone, `plan_tier` changed to `premium`

**Dashboard**: Go to [Customers](https://dashboard.stripe.com/test/customers) and click a customer to see its metadata tab.

### Metadata Best Practices

1. Use consistent key names across your application
2. Prefix keys by domain (`order_`, `user_`, `campaign_`)
3. Never store sensitive data (PII, passwords, secrets)
4. Use it for cross-referencing with your internal systems

---

## Stripe Testing Tools: Overview

The next two notebooks cover the remaining tools:

| Tool | What it does |
|------|--------------|
| **Sandboxes** | Isolated environments per team or project |
| **Test Cards** | Simulate successful payments, declines, and specific errors |
| **Test Clocks** | Fast-forward time to test subscription trials and renewals |

## Summary

- **Sandboxes** let you test safely without affecting live data
- Always use **`sk_test_`** keys in sandboxes
- **`expand`** reduces API calls by embedding related objects
- **`metadata`** stores custom key-value data on any Stripe object

## Next Steps

Open `02_sandbox_and_test_cards.ipynb` to simulate payments with test cards.